# Silver — Tabela de custos tratada e classificada

Lê `bronze.custo_soja_raw`, aplica as transformações da camada Silver e faz o
**join com `silver.dim_classificacao`** para atribuir o grupo econômico a cada
linha. Grava `silver.custo_soja`.

**Pré-requisito:** `bronze.custo_soja_raw` e `silver.dim_classificacao` já
existem (notebooks 01 e 02).

Transformações:
- Separa número da descrição; gera `descricao` limpa e `descricao_canonica`.
- Tipa `custo_ha` e `custo_60kg` como numéricos (vazio → NULL).
- Normaliza `produtividade_kg_ha` a partir do texto cru (trata as várias eras).
- Limpa `categoria` (remove o romano "I - ").
- Classifica `tipo_linha` em `item` ou `total`.
- Junta `grupo_economico` da dimensão (não casou → 'Nao classificado').

**Idempotente:** recria `silver.custo_soja` a cada execução.

## 1. Imports e leitura das fontes

In [2]:
import re 
import unicodedata
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from db import get_engine


engine = get_engine()

df = pd.read_sql("SELECT * FROM bronze.custo_soja_raw", engine)
dim = pd.read_sql("SELECT descricao_canonica, grupo_economico FROM silver.dim_classificacao", engine)
print(f"Bronze: {len(df):,} linhas | Dimensão: {len(dim)} descrições")

Bronze: 23,727 linhas | Dimensão: 115 descrições


## 2. Funções de transformação

O extrator de produtividade trata as duas formas em que o valor aparece no
arquivo: número brasileiro embutido no texto (`3.660,00 kg/ha` → vírgula decimal)
e valor em célula separada capturado após `|` (era antiga, ponto decimal).

In [5]:
RE_NUM_PREFIX = re.compile(r"^\s*\d+(\.\d+)?\s*-\s*")
RE_TOTAL = re.compile(r"(TOTAL|CUSTO VAR|CUSTO OPER|CUSTO FIXO|CUSTO TOTAL|RECEITA)", re.I)
RE_ROMANO = re.compile(r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X)\s*-\s*", re.I)


def normaliza(desc):
    """(descricao_limpa, descricao_canonica) — igual ao notebook 02."""
    s = str(desc).replace("\t", "")
    m = RE_NUM_PREFIX.match(s)
    if m:
        s = s[m.end():]
    limpa = s.strip().rstrip(":").strip()
    canon = unicodedata.normalize("NFKD", limpa.lower())
    canon = "".join(c for c in canon if not unicodedata.combining(c))
    canon = re.sub(r"\s+", " ", canon).strip()
    return limpa, canon


def extrai_produtividade(raw):
    """Número de produtividade a partir do texto cru, tratando as várias eras."""
    if raw is None or str(raw).strip() == "":
        return None
    s = str(raw)
    # Caso 1: valor em célula separada, após '|' (ponto decimal)
    if "|" in s:
        depois = s.split("|")[-1].strip()
        m = re.search(r"\d+(?:\.\d+)?", depois)
        if m:
            try:
                v = float(m.group(0))
                if 0 < v < 20000:
                    return v
            except ValueError:
                pass
    # Caso 2: número brasileiro embutido (ponto=milhar, vírgula=decimal)
    m = re.search(r"\d[\d.]*(?:,\d+)?", s)
    if m:
        try:
            v = float(m.group(0).replace(".", "").replace(",", "."))
            if 0 < v < 20000:
                return v
        except ValueError:
            pass
    return None


def limpa_categoria(cat):
    if cat is None:
        return None
    s = RE_ROMANO.sub("", str(cat))
    return s.strip().title()

## 3. Aplicar transformações

In [6]:
s = df.copy()

# descrição limpa + canônica
s[["descricao", "descricao_canonica"]] = s["descricao_custo"].apply(
    lambda x: pd.Series(normaliza(x))
)

# valores tipados (string vazia / lixo -> NaN -> NULL no banco)
s["custo_ha"] = pd.to_numeric(s["(R$/ha)"], errors="coerce")
s["custo_60kg"] = pd.to_numeric(s["(R$/60 kg)"], errors="coerce")

# produtividade normalizada
s["produtividade_kg_ha"] = s["produtividade_media(kg/ha)"].apply(extrai_produtividade)

# categoria limpa
s["categoria"] = s["categoria"].apply(limpa_categoria)

# tipo de linha
s["tipo_linha"] = s["descricao_custo"].apply(
    lambda d: "total" if RE_TOTAL.search(str(d)) else "item"
)

print("Transformações aplicadas.")
print("produtividade nula:", s["produtividade_kg_ha"].isna().sum())
print("custo_ha nulo:", s["custo_ha"].isna().sum())

Transformações aplicadas.
produtividade nula: 0
custo_ha nulo: 830


## 4. Join com a dimensão de classificação

Atribui `grupo_economico` a cada linha. O que não casar recebe `Nao classificado`
— o esperado é que sejam apenas os totais (que a dimensão marca assim de
propósito).

In [7]:
s = s.merge(dim, on="descricao_canonica", how="left")
s["grupo_economico"] = s["grupo_economico"].fillna("Nao classificado")

# sanidade: todo 'Nao classificado' deveria ser 'total'
nc_itens = s[(s["grupo_economico"] == "Nao classificado") & (s["tipo_linha"] == "item")]
print(f"Itens (não-total) sem classificação: {len(nc_itens)}")
if len(nc_itens):
    print("  Descrições órfãs:", nc_itens["descricao_canonica"].unique()[:10])
else:
    print("  OK — só os totais ficaram como 'Nao classificado'.")

Itens (não-total) sem classificação: 0
  OK — só os totais ficaram como 'Nao classificado'.


## 5. Selecionar colunas e gravar `silver.custo_soja`

In [8]:
COLUNAS = [
    "local", "estado", "ano", "tipo_cultivo",
    "categoria", "descricao", "descricao_canonica", "grupo_economico",
    "custo_ha", "custo_60kg", "produtividade_kg_ha", "tipo_linha",
    "aba_origem", "linha_origem",
]
silver = s[COLUNAS].copy()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS silver.custo_soja;"))

silver.to_sql(
    name="custo_soja", schema="silver", con=engine,
    if_exists="replace", index=False, chunksize=5000, method="multi",
)
print(f"Gravado silver.custo_soja: {len(silver):,} linhas")

Gravado silver.custo_soja: 23,727 linhas
